# NB46 — PAH ve CFTR: Reverse-Distribution Egitim Deneyi

**Hipotez:** Reverse-distribution (gercek %60 benign / %40 pathogenic resample ile egitim),
MASTER (NB39: 0.62->0.638) ve KANSER'de (NB32: 0.69->0.73) kanitlanmis en guclu tek kaldirac.
Bu teknik PAH ve CFTR panellerine hic uygulanmadi (bkz. reports/literature_research_panel_improvements_2026-07-24.md, S2 yol haritasi #2).

**Mevcut sampiyonlar (degistirilmeyecek referans):**
| Panel | Sampiyon | Boot-F1 / LOO-metrik | Kaynak |
|---|---|---|---|
| PAH | P4_COMBINED_BalBag | Boot-mean=0.582 (MCC=0.529) | NB21 |
| CFTR | S0c_COMBINED (+prior-shift) | Boot-mean=0.863 (LOO-MCC=0.644, prec=1.0) | NB20 |

**Bu notebook'ta degisen TEK degisken:** COMBINED egitim havuzunun benign:pathogenic
kompozisyonu. Degerlendirme protokolu (LOO-CV + robust bootstrap %80/20 threshold + prior-shift)
NB21/NB20 ile BIREBIR AYNI tutuluyor -- karsilastirilabilirlik icin.

**Senaryolar (her panel icin):**
| # | Senaryo | Kompozisyon |
|---|---|---|
| R0 | COMBINED_baseline | Orijinal COMBINED dagilimi (kontrol, NB21/NB20 ile ayni) |
| R1 | COMBINED_REVERSE_6040 | COMBINED'dan %60 benign / %40 patho gercek resample |
| R2 | COMBINED_REVERSE_7030 | COMBINED'dan %70 benign / %30 patho gercek resample |
| R3 | COMBINED_REVERSE_8020 | COMBINED'dan %80 benign / %20 patho gercek resample (agresif) |

Her senaryoda hem raw hem prior-shift-adjusted metrikler raporlanir. FLOOR-F1 referansi
(kendi LOO test havuzunun prevalansindan) her satirda yan yana verilir (CLAUDE.md zorunlu kural).

In [ ]:
# Cell 1: Imports & Config
import os, sys, warnings, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from collections import OrderedDict

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

PROJECT_ROOT = os.path.dirname(os.path.abspath("__file__"))
if not os.path.exists(os.path.join(PROJECT_ROOT, "config.py")):
    PROJECT_ROOT = os.path.dirname(os.getcwd())
if not os.path.exists(os.path.join(PROJECT_ROOT, "config.py")):
    PROJECT_ROOT = os.getcwd()
    while PROJECT_ROOT != "/" and not os.path.exists(os.path.join(PROJECT_ROOT, "config.py")):
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

sys.path.insert(0, PROJECT_ROOT)

from config import SEED, REPORTS_DIR
import src.columns_real as CR
from src.metrics import compute_all_metrics

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    matthews_corrcoef, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix
)
from lightgbm import LGBMClassifier
from imblearn.ensemble import BalancedBaggingClassifier

np.random.seed(SEED)

# Sabitler (NB21/NB20 ile birebir ayni)
PI_TEST = 0.20
FINAL_BENIGN_FRAC = 0.80
N_BOOT = 50
BOOT_SEED = 123
N_ROBUST = 50

DATA_DIR = os.path.join(PROJECT_ROOT, "data", "real_data")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results", "v27_reverse_distribution_pah_cftr")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

ID_COL = CR.ID_COL
TARGET = CR.TARGET_COL

LGBM_PARAMS = {
    "n_estimators": 300,
    "num_leaves": 31,
    "learning_rate": 0.05,
    "min_child_samples": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": SEED,
    "verbose": -1,
    "n_jobs": -1,
}

def lgbm_classifier(**kw):
    params = {**LGBM_PARAMS, **kw}
    return LGBMClassifier(**params)

def lgbm_classifier_single_thread(**kw):
    """BalancedBaggingClassifier base estimator icin: dis paralellik (n_jobs=-1
    BalancedBagging'de) ile ic paralellik (LGBM'in kendi n_jobs=-1'i) macOS'ta
    loky+LightGBM nested parallelism deadlock'una yol aciyor. Base estimator
    tek-thread olmali; disaridaki BalancedBagging paralelligi yeterli."""
    params = {**LGBM_PARAMS, **kw}
    params["n_jobs"] = 1
    return LGBMClassifier(**params)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"SEED={SEED}, PI_TEST={PI_TEST}, N_BOOT={N_BOOT}")
print(f"Results -> {RESULTS_DIR}")

In [ ]:
# Cell 2: Veri Yukleme + Ortak Sutun Temizligi (4 panel)
def load_panel(name):
    return pd.read_csv(os.path.join(DATA_DIR, CR.PANEL_INFO[name]["file"]))

df_master = load_panel("MASTER")
df_kanser = load_panel("KANSER")
df_cftr = load_panel("CFTR")
df_pah = load_panel("PAH")

print(f"MASTER: {df_master.shape} (pos={df_master[TARGET].sum()}, neg={(df_master[TARGET]==0).sum()})")
print(f"KANSER: {df_kanser.shape} (pos={df_kanser[TARGET].sum()}, neg={(df_kanser[TARGET]==0).sum()})")
print(f"CFTR:   {df_cftr.shape}   (pos={df_cftr[TARGET].sum()}, neg={(df_cftr[TARGET]==0).sum()})")
print(f"PAH:    {df_pah.shape}  (pos={df_pah[TARGET].sum()}, neg={(df_pah[TARGET]==0).sum()})")

feat_cols = [c for c in df_master.columns if c not in [ID_COL, TARGET]]

# Sabit ve ozdes sutun temizligi (MASTER uzerinde tespit, NB21/NB39 ile ayni mantik)
constant_cols = CR.get_constant_cols(df_master)
dup_pairs = CR.get_duplicate_col_pairs(df_master)
dup_drop = set(c2 for c1, c2 in dup_pairs)
drop_cols = set(constant_cols) | dup_drop
print(f"\nConstant: {len(constant_cols)}, Duplicate pairs: {len(dup_pairs)} -> drop {len(dup_drop)}")
print(f"Toplam drop: {len(drop_cols)}, Kalan feature: {len(feat_cols) - len(drop_cols)}")

keep_cols = [c for c in feat_cols if c not in drop_cols]
df_master = df_master[[ID_COL, TARGET] + keep_cols].copy()
df_kanser = df_kanser[[ID_COL, TARGET] + keep_cols].copy()
df_cftr = df_cftr[[ID_COL, TARGET] + keep_cols].copy()
df_pah = df_pah[[ID_COL, TARGET] + keep_cols].copy()

# Cross-panel birebir-ayni satir drop (NB15/NB21 dogrulanmis yaklasim)
def find_exact_dups(panel_df, master_df, feat_cols, target):
    common_ids = set(panel_df[ID_COL]) & set(master_df[ID_COL])
    if not common_ids:
        return []
    dup_ids = []
    check_cols = feat_cols + [target]
    for vid in common_ids:
        p_row = panel_df.loc[panel_df[ID_COL] == vid, check_cols].iloc[0]
        m_rows = master_df.loc[master_df[ID_COL] == vid, check_cols]
        for _, m_row in m_rows.iterrows():
            if p_row.equals(m_row):
                dup_ids.append(vid)
                break
    return dup_ids

for name, df_panel in [("KANSER", df_kanser), ("CFTR", df_cftr), ("PAH", df_pah)]:
    dup_ids = find_exact_dups(df_panel, df_master, keep_cols, TARGET)
    if dup_ids:
        print(f"{name}: {len(dup_ids)} birebir-ayni satir drop edilecek")

dup_ids_kanser = find_exact_dups(df_kanser, df_master, keep_cols, TARGET)
dup_ids_cftr = find_exact_dups(df_cftr, df_master, keep_cols, TARGET)
dup_ids_pah = find_exact_dups(df_pah, df_master, keep_cols, TARGET)

df_kanser = df_kanser[~df_kanser[ID_COL].isin(dup_ids_kanser)].reset_index(drop=True)
df_cftr = df_cftr[~df_cftr[ID_COL].isin(dup_ids_cftr)].reset_index(drop=True)
df_pah = df_pah[~df_pah[ID_COL].isin(dup_ids_pah)].reset_index(drop=True)

print(f"\nFinal shapes: MASTER={df_master.shape}, KANSER={df_kanser.shape}, CFTR={df_cftr.shape}, PAH={df_pah.shape}")

In [ ]:
# Cell 3: Degerlendirme Altyapisi -- LOO-CV metrikleri, robust bootstrap %80/20, prior-shift
# (NB21/NB20 ile birebir ayni fonksiyonlar -- karsilastirilabilirlik icin degistirilmedi)

def adjust_prior_shift(proba, pi_train, pi_test=PI_TEST):
    proba = np.clip(proba, 1e-7, 1 - 1e-7)
    odds = proba / (1 - proba)
    R = (pi_test / (1 - pi_test)) / (pi_train / (1 - pi_train))
    odds_adj = odds * R
    return odds_adj / (1 + odds_adj)

def _f1_pos(y, p):
    return f1_score(y, p, pos_label=1, zero_division=0)

def _resample_8020(y, prob, rng):
    y = np.asarray(y)
    prob = np.asarray(prob)
    neg = np.where(y == 0)[0]
    pos = np.where(y == 1)[0]
    if len(neg) == 0 or len(pos) == 0:
        return y, prob
    npos = max(1, int(round(len(neg) * (1 - FINAL_BENIGN_FRAC) / FINAL_BENIGN_FRAC)))
    keep = np.concatenate([neg, rng.choice(pos, size=npos, replace=True)])
    return y[keep], prob[keep]

def bootstrap_8020(y_te, p_te, thr, n=N_BOOT):
    rng = np.random.RandomState(BOOT_SEED)
    f1s = []
    for _ in range(n):
        yb, pb = _resample_8020(y_te, p_te, rng)
        f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
    f1s = np.array(f1s)
    return {"mean": float(f1s.mean()), "std": float(f1s.std()),
            "lo": float(np.percentile(f1s, 2.5)), "hi": float(np.percentile(f1s, 97.5))}

def select_threshold_8020_robust(y, prob, n=N_ROBUST):
    rng = np.random.RandomState(BOOT_SEED)
    thr_scores = {}
    for thr in np.arange(0.05, 0.95, 0.01):
        thr = round(thr, 2)
        f1s = []
        for _ in range(n):
            yb, pb = _resample_8020(y, prob, rng)
            f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
        thr_scores[thr] = np.mean(f1s)
    best_thr = max(thr_scores, key=thr_scores.get)
    return float(best_thr)

def floor_f1(y_true):
    "Trivial 'hep pathogenic tahmin et' baseline F1 = 2*prev/(1+prev), test havuzunun kendi prevalansindan."
    prev = float(np.mean(y_true))
    return 2 * prev / (1 + prev)

def loo_metrics(y_true, oof_proba, prior_shift=False, pi_train=None):
    if pi_train is None:
        pi_train = y_true.mean()
    prob = adjust_prior_shift(oof_proba, pi_train=pi_train) if prior_shift else oof_proba
    thr = select_threshold_8020_robust(y_true, prob)
    y_pred = (prob >= thr).astype(int)
    mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else 0.0
    f1 = _f1_pos(y_true, y_pred)
    auc = roc_auc_score(y_true, prob) if len(np.unique(y_true)) > 1 else 0.0
    prec = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    rec = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    boot = bootstrap_8020(y_true, prob, thr)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {"mcc": mcc, "f1": f1, "auc": auc, "precision": prec, "recall": rec,
            "thr": thr, "boot8020": boot,
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}

def train_metrics_at(y_train, p_train, thr):
    yp = (p_train >= thr).astype(int)
    return {
        "train_f1": float(_f1_pos(y_train, yp)),
        "train_mcc": float(matthews_corrcoef(y_train, yp)),
    }

def le_encode(X_df):
    Xn = X_df.copy()
    cat_cols = Xn.select_dtypes(include=["object", "category"]).columns.tolist()
    for c in cat_cols:
        Xn[c] = Xn[c].fillna("MISSING").astype(str)
        le = LabelEncoder()
        Xn[c] = le.fit_transform(Xn[c])
    return Xn

print("Degerlendirme altyapisi hazir (NB21/NB20 ile birebir ayni).")

In [ ]:
# Cell 4: Reverse-Distribution Yardimci Fonksiyonu (NB39 deseni)

def make_reversed_subset(X_pool, y_pool, benign_frac, seed=SEED):
    """Egitim havuzundan gercek resample ile hedef benign/pathogenic oranini uret.
    NB39 ile ayni mantik: dusuk benign_frac hedeflerinde tum benign'i al, patho'yu kis;
    yuksek (>=0.80) hedeflerde benign'den de kisarak agresif oran uygula."""
    idx_ben = np.where(y_pool.values == 0)[0]
    idx_pat = np.where(y_pool.values == 1)[0]
    rng = np.random.RandomState(seed)

    if benign_frac >= 0.80:
        n_pat = min(len(idx_pat), int(len(idx_ben) * (1 - benign_frac) / benign_frac))
        n_ben = int(n_pat * benign_frac / (1 - benign_frac))
        n_ben = min(n_ben, len(idx_ben))
        ben_sample = rng.choice(idx_ben, size=n_ben, replace=False)
        pat_sample = rng.choice(idx_pat, size=n_pat, replace=False)
    else:
        n_ben = len(idx_ben)
        n_pat = int(n_ben * (1 - benign_frac) / benign_frac)
        n_pat = min(n_pat, len(idx_pat))
        ben_sample = idx_ben
        pat_sample = rng.choice(idx_pat, size=n_pat, replace=False)

    idx = np.concatenate([ben_sample, pat_sample])
    rng.shuffle(idx)
    return X_pool.iloc[idx].copy(), y_pool.iloc[idx].copy()

# Dogrulama testi
_X_test_dummy = df_master[keep_cols]
_y_test_dummy = df_master[TARGET]
for name, frac in [("R1_6040", 0.60), ("R2_7030", 0.70), ("R3_8020", 0.80)]:
    Xs, ys = make_reversed_subset(_X_test_dummy, _y_test_dummy, frac)
    print(f"{name}: n={len(ys)}, benign_frac={(ys==0).mean():.3f} (hedef={frac}), pos={ys.sum()}, neg={(ys==0).sum()}")

In [ ]:
# Cell 5: Panel Deney Calistirici -- bir panel icin 4 senaryo x (LGBM + BalBag) calistirir

def run_panel_experiment(panel_name, df_target, df_other_panels):
    """
    panel_name: 'PAH' veya 'CFTR'
    df_target: hedef panel dataframe (LOO-CV test edilecek, egitime GIRMEZ)
    df_other_panels: diger panellerin listesi (COMBINED havuzu olusturur)
    """
    print("=" * 70)
    print(f"PANEL: {panel_name}")
    print("=" * 70)

    df_combined = pd.concat(df_other_panels, ignore_index=True)
    X_combined_raw = df_combined[keep_cols]
    y_combined = df_combined[TARGET]
    X_target_raw = df_target[keep_cols]
    y_target = df_target[TARGET]

    X_combined_le = le_encode(X_combined_raw)
    X_target_le = le_encode(X_target_raw)

    print(f"COMBINED havuzu: n={len(y_combined)} (pos={y_combined.sum()}, neg={(y_combined==0).sum()}, "
          f"benign_frac={(y_combined==0).mean():.3f})")
    print(f"Hedef panel ({panel_name}): n={len(y_target)} (pos={y_target.sum()}, neg={(y_target==0).sum()})")

    floor = floor_f1(y_target.values)
    print(f"FLOOR-F1 (hedef panel prevalansindan): {floor:.4f}")

    scenarios = OrderedDict([
        ("R0_baseline", None),
        ("R1_REVERSE_6040", 0.60),
        ("R2_REVERSE_7030", 0.70),
        ("R3_REVERSE_8020", 0.80),
    ])

    panel_results = OrderedDict()

    for scn_name, benign_frac in scenarios.items():
        if benign_frac is None:
            X_scn, y_scn = X_combined_le, y_combined
        else:
            X_scn_raw, y_scn = make_reversed_subset(X_combined_raw, y_combined, benign_frac)
            X_scn = le_encode(X_scn_raw)

        pi_train = float(y_scn.mean())
        n_train = len(y_scn)
        print(f"\n--- {scn_name}: n_train={n_train}, benign_frac={(y_scn==0).mean():.3f}, pi_train={pi_train:.3f} ---")

        # Model 1: LGBM (class_weight balanced)
        m_lgbm = lgbm_classifier(class_weight="balanced")
        m_lgbm.fit(X_scn, y_scn)
        p_train_lgbm = m_lgbm.predict_proba(X_scn)[:, 1]
        p_target_lgbm = m_lgbm.predict_proba(X_target_le)[:, 1]

        thr_lgbm = select_threshold_8020_robust(y_scn, p_train_lgbm)
        train_lgbm = train_metrics_at(y_scn, p_train_lgbm, thr_lgbm)
        loo_lgbm_raw = loo_metrics(y_target, p_target_lgbm, prior_shift=False)
        loo_lgbm_prior = loo_metrics(y_target, p_target_lgbm, prior_shift=True, pi_train=pi_train)

        panel_results[f"{scn_name}/lgbm"] = {
            "scenario": scn_name, "model": "lgbm", "n_train": n_train, "pi_train": pi_train,
            "loo_raw": loo_lgbm_raw, "loo_prior": loo_lgbm_prior, "train": train_lgbm,
            "floor_f1": floor,
        }
        print(f"  [lgbm]   MCC(raw)={loo_lgbm_raw['mcc']:.4f} MCC(prior)={loo_lgbm_prior['mcc']:.4f} "
              f"Boot-mean(prior)={loo_lgbm_prior['boot8020']['mean']:.4f} floor={floor:.4f}")

        # Model 2: BalancedBagging (LGBM base)
        m_bb = BalancedBaggingClassifier(
            estimator=lgbm_classifier_single_thread(),
            n_estimators=20,
            sampling_strategy="not minority",
            random_state=SEED,
            n_jobs=-1,
        )
        m_bb.fit(X_scn, y_scn)
        p_train_bb = m_bb.predict_proba(X_scn)[:, 1]
        p_target_bb = m_bb.predict_proba(X_target_le)[:, 1]

        thr_bb = select_threshold_8020_robust(y_scn, p_train_bb)
        train_bb = train_metrics_at(y_scn, p_train_bb, thr_bb)
        loo_bb_raw = loo_metrics(y_target, p_target_bb, prior_shift=False)
        loo_bb_prior = loo_metrics(y_target, p_target_bb, prior_shift=True, pi_train=pi_train)

        panel_results[f"{scn_name}/balbag"] = {
            "scenario": scn_name, "model": "balbag", "n_train": n_train, "pi_train": pi_train,
            "loo_raw": loo_bb_raw, "loo_prior": loo_bb_prior, "train": train_bb,
            "floor_f1": floor,
        }
        print(f"  [balbag] MCC(raw)={loo_bb_raw['mcc']:.4f} MCC(prior)={loo_bb_prior['mcc']:.4f} "
              f"Boot-mean(prior)={loo_bb_prior['boot8020']['mean']:.4f} floor={floor:.4f}")

    return panel_results, floor

print("Panel deney calistirici hazir.")

In [ ]:
# Cell 6: PAH Deneyi Calistir (COMBINED = MASTER+KANSER+CFTR, PAH haric)
pah_results, pah_floor = run_panel_experiment(
    "PAH", df_pah, [df_master, df_kanser, df_cftr]
)

In [ ]:
# Cell 7: CFTR Deneyi Calistir (COMBINED = MASTER+KANSER+PAH, CFTR haric)
cftr_results, cftr_floor = run_panel_experiment(
    "CFTR", df_cftr, [df_master, df_kanser, df_pah]
)

In [ ]:
# Cell 8: Sonuc Derleme -- Karsilastirma Tablolari (referans sampiyonlarla yan yana)

def build_comparison_df(panel_results, panel_name, floor, champion_boot, champion_name):
    rows = []
    for key, res in panel_results.items():
        loo_p = res["loo_prior"]
        loo_r = res["loo_raw"]
        boot = loo_p["boot8020"]
        rows.append({
            "Panel": panel_name,
            "Deney": key,
            "Senaryo": res["scenario"],
            "Model": res["model"],
            "n_train": res["n_train"],
            "benign_frac_train": round(1 - res["pi_train"], 3),
            "LOO-MCC(raw)": round(loo_r["mcc"], 4),
            "LOO-MCC(prior)": round(loo_p["mcc"], 4),
            "Boot-F1(prior)": round(boot["mean"], 4),
            "Boot-std": round(boot["std"], 4),
            "Boot-CI": f"[{boot['lo']:.3f}-{boot['hi']:.3f}]",
            "Precision": round(loo_p["precision"], 4),
            "Recall": round(loo_p["recall"], 4),
            "FLOOR-F1": round(floor, 4),
            "Gecti_mi_Floor": "EVET" if boot["mean"] > floor else "HAYIR",
            "TP": loo_p["tp"], "FP": loo_p["fp"], "FN": loo_p["fn"], "TN": loo_p["tn"],
            "Train-F1": round(res["train"]["train_f1"], 4),
        })
    df_cmp = pd.DataFrame(rows)
    # Referans sampiyon satiri ekle
    ref_row = {
        "Panel": panel_name, "Deney": f"REF_{champion_name}", "Senaryo": "REF", "Model": "REF",
        "n_train": None, "benign_frac_train": None,
        "LOO-MCC(raw)": None, "LOO-MCC(prior)": None,
        "Boot-F1(prior)": champion_boot, "Boot-std": None, "Boot-CI": "",
        "Precision": None, "Recall": None, "FLOOR-F1": round(floor, 4),
        "Gecti_mi_Floor": "EVET" if champion_boot > floor else "HAYIR",
        "TP": None, "FP": None, "FN": None, "TN": None, "Train-F1": None,
    }
    df_cmp = pd.concat([df_cmp, pd.DataFrame([ref_row])], ignore_index=True)
    return df_cmp.sort_values("Boot-F1(prior)", ascending=False).reset_index(drop=True)

pah_cmp = build_comparison_df(pah_results, "PAH", pah_floor, champion_boot=0.582, champion_name="P4_COMBINED_BalBag_NB21")
cftr_cmp = build_comparison_df(cftr_results, "CFTR", cftr_floor, champion_boot=0.863, champion_name="S0c_PriorShift_NB20")

print("\n=== PAH: Reverse-Distribution Sonuclari ===")
print(pah_cmp.to_string(index=False))

print("\n=== CFTR: Reverse-Distribution Sonuclari ===")
print(cftr_cmp.to_string(index=False))

pah_cmp.to_csv(os.path.join(RESULTS_DIR, "pah_reverse_distribution_results.csv"), index=False)
cftr_cmp.to_csv(os.path.join(RESULTS_DIR, "cftr_reverse_distribution_results.csv"), index=False)

pah_best = pah_cmp[pah_cmp["Senaryo"] != "REF"].iloc[0]
cftr_best = cftr_cmp[cftr_cmp["Senaryo"] != "REF"].iloc[0]

print(f"\n→ PAH en iyi reverse-dist: {pah_best['Deney']} Boot-F1={pah_best['Boot-F1(prior)']:.4f} "
      f"(referans NB21={0.582:.4f}, delta={pah_best['Boot-F1(prior)']-0.582:+.4f}, floor={pah_floor:.4f})")
print(f"→ CFTR en iyi reverse-dist: {cftr_best['Deney']} Boot-F1={cftr_best['Boot-F1(prior)']:.4f} "
      f"(referans NB20={0.863:.4f}, delta={cftr_best['Boot-F1(prior)']-0.863:+.4f}, floor={cftr_floor:.4f})")

In [ ]:
# Cell 9: Gorsellestirmeler

fig, axes = plt.subplots(2, 2, figsize=(15, 11))

for ax_row, (cmp_df, panel_name, ref_boot) in zip(
    axes, [(pah_cmp, "PAH", 0.582), (cftr_cmp, "CFTR", 0.863)]
):
    plot_df = cmp_df[cmp_df["Senaryo"] != "REF"]
    scenarios_order = ["R0_baseline", "R1_REVERSE_6040", "R2_REVERSE_7030", "R3_REVERSE_8020"]

    # Sol: senaryo x model heatmap
    ax = ax_row[0]
    heatmap_data = plot_df.pivot_table(index="Senaryo", columns="Model", values="Boot-F1(prior)")
    heatmap_data = heatmap_data.reindex(scenarios_order)
    sns.heatmap(heatmap_data.astype(float), annot=True, fmt=".4f", cmap="RdYlGn", ax=ax, linewidths=0.5)
    ax.set_title(f"{panel_name}: Boot-F1(prior) — Senaryo x Model")

    # Sag: bar chart + floor + referans cizgisi
    ax = ax_row[1]
    plot_df_sorted = plot_df.copy()
    plot_df_sorted["label"] = plot_df_sorted["Senaryo"] + "/" + plot_df_sorted["Model"]
    ax.bar(plot_df_sorted["label"], plot_df_sorted["Boot-F1(prior)"], color="steelblue")
    ax.axhline(y=ref_boot, color="red", linestyle="--", label=f"Referans sampiyon={ref_boot:.3f}")
    floor_val = plot_df_sorted["FLOOR-F1"].iloc[0]
    ax.axhline(y=floor_val, color="orange", linestyle=":", label=f"Floor-F1={floor_val:.3f}")
    ax.set_title(f"{panel_name}: Boot-F1(prior) vs Referans ve Floor")
    ax.set_xticklabels(plot_df_sorted["label"], rotation=45, ha="right", fontsize=8)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "reverse_distribution_pah_cftr.png"), dpi=150, bbox_inches="tight")
plt.close()
print(f"Gorsellestirme kaydedildi: {RESULTS_DIR}/reverse_distribution_pah_cftr.png")

In [ ]:
# Cell 10: Kapsamli PDF Rapor
from fpdf import FPDF

class NB46Report(FPDF):
    def header(self):
        self.set_font("Helvetica", "B", 14)
        self.cell(0, 10, "NB46: PAH ve CFTR - Reverse-Distribution Egitim Deneyi", 0, 1, "C")
        self.set_font("Helvetica", "", 9)
        self.cell(0, 5, f"SEED={SEED} | PI_TEST={PI_TEST} | LOO-CV + Robust Bootstrap 80/20 Protokolu", 0, 1, "C")
        self.ln(3)

    def footer(self):
        self.set_y(-15)
        self.set_font("Helvetica", "I", 8)
        self.cell(0, 10, f"Sayfa {self.page_no()}", 0, 0, "C")

    def section_title(self, title):
        self.set_font("Helvetica", "B", 12)
        self.set_fill_color(41, 128, 185)
        self.set_text_color(255, 255, 255)
        self.cell(0, 8, f"  {title}", 0, 1, "L", fill=True)
        self.set_text_color(0, 0, 0)
        self.ln(2)

    def sub_title(self, title):
        self.set_font("Helvetica", "B", 10)
        self.set_text_color(41, 128, 185)
        self.cell(0, 6, title, 0, 1, "L")
        self.set_text_color(0, 0, 0)
        self.ln(1)

    def body_text(self, text):
        self.set_font("Helvetica", "", 9)
        self.multi_cell(0, 5, text)
        self.ln(2)

    def add_table(self, headers, rows, col_widths=None):
        if col_widths is None:
            col_widths = [self.epw / len(headers)] * len(headers)
        self.set_font("Helvetica", "B", 7)
        self.set_fill_color(220, 220, 220)
        for h, w in zip(headers, col_widths):
            self.cell(w, 6, str(h), 1, 0, "C", fill=True)
        self.ln()
        self.set_font("Helvetica", "", 7)
        for row in rows:
            for val, w in zip(row, col_widths):
                self.cell(w, 5.5, str(val), 1, 0, "C")
            self.ln()
        self.ln(3)


pdf = NB46Report(orientation="L", format="A4")
pdf.set_auto_page_break(auto=True, margin=15)
pdf.add_page()

# --- Yonetici Ozeti ---
pdf.section_title("Yonetici Ozeti")
pdf.body_text(
    "Amac: Literatur arastirmasinda (reports/literature_research_panel_improvements_2026-07-24.md) "
    "en yuksek getirili/en dusuk riskli hamle olarak belirlenen reverse-distribution (gercek %60/40 "
    "benign/pathogenic resample ile egitim) tekniginin PAH ve CFTR panellerine uygulanmasi. Bu teknik "
    "MASTER (NB39: 0.638) ve KANSER'de (NB32: 0.730) kanitlanmis ama PAH ve CFTR'de hic denenmemisti.\n\n"
    "Yontem: Her panel icin COMBINED egitim havuzu (diger 3 panelin birlesimi, hedef panel HARIC) "
    "olusturuldu. Bu havuzdan 4 senaryo turetildi: R0 (orijinal dagilim, kontrol), R1 (%60 benign/%40 "
    "patho), R2 (%70/%30), R3 (%80/%20 agresif). Her senaryoda LGBM ve BalancedBagging modelleri "
    "egitildi. Degerlendirme NB21 (PAH)/NB20 (CFTR) ile BIREBIR AYNI protokolde yapildi: hedef panel "
    "uzerinde LOO-CV, robust N=50 bootstrap %80/20 threshold secimi, Saerens prior-shift, ve floor-F1 "
    "referansi (kendi test havuzunun prevalansindan)."
)

pdf.sub_title("Ana Bulgular")
pah_delta = pah_best["Boot-F1(prior)"] - 0.582
cftr_delta = cftr_best["Boot-F1(prior)"] - 0.863
pdf.body_text(
    f"PAH: En iyi reverse-dist senaryosu {pah_best['Deney']} ile Boot-F1(prior)={pah_best['Boot-F1(prior)']:.4f} "
    f"elde edildi (referans NB21 sampiyonu P4_COMBINED_BalBag=0.582, delta={pah_delta:+.4f}). "
    f"Floor-F1={pah_floor:.4f}.\n"
    f"CFTR: En iyi reverse-dist senaryosu {cftr_best['Deney']} ile Boot-F1(prior)={cftr_best['Boot-F1(prior)']:.4f} "
    f"elde edildi (referans NB20 sampiyonu S0c_PriorShift=0.863, delta={cftr_delta:+.4f}). "
    f"Floor-F1={cftr_floor:.4f}."
)

# --- PAH Tablosu ---
pdf.add_page()
pdf.section_title("PAH - Reverse-Distribution Sonuclari")
pdf.body_text(
    "COMBINED havuzu = MASTER + KANSER + CFTR (PAH panelin kendisi egitime girmiyor, LOO-CV ile test ediliyor)."
)
headers = ["Deney", "n_train", "benign_frac", "LOO-MCC(prior)", "Boot-F1(prior)", "Boot-CI", "Precision", "Recall", "FLOOR-F1", "Floor Gecti mi"]
col_widths = [45, 18, 20, 22, 22, 30, 18, 18, 18, 22]
rows = []
for _, r in pah_cmp.iterrows():
    rows.append([
        r["Deney"], r["n_train"] if pd.notna(r["n_train"]) else "-",
        r["benign_frac_train"] if pd.notna(r["benign_frac_train"]) else "-",
        r["LOO-MCC(prior)"] if pd.notna(r["LOO-MCC(prior)"]) else "-",
        f"{r['Boot-F1(prior)']:.4f}", r["Boot-CI"],
        r["Precision"] if pd.notna(r["Precision"]) else "-",
        r["Recall"] if pd.notna(r["Recall"]) else "-",
        r["FLOOR-F1"], r["Gecti_mi_Floor"],
    ])
pdf.add_table(headers, rows, col_widths)

# --- CFTR Tablosu ---
pdf.section_title("CFTR - Reverse-Distribution Sonuclari")
pdf.body_text(
    "COMBINED havuzu = MASTER + KANSER + PAH (CFTR panelin kendisi egitime girmiyor, LOO-CV ile test ediliyor). "
    "UYARI: CFTR n=21 benign -> tum karsilastirmalar dusuk-guven bandinda (CLAUDE.md 'CFTR DEGERLENDIRME GUVENI YOK')."
)
rows = []
for _, r in cftr_cmp.iterrows():
    rows.append([
        r["Deney"], r["n_train"] if pd.notna(r["n_train"]) else "-",
        r["benign_frac_train"] if pd.notna(r["benign_frac_train"]) else "-",
        r["LOO-MCC(prior)"] if pd.notna(r["LOO-MCC(prior)"]) else "-",
        f"{r['Boot-F1(prior)']:.4f}", r["Boot-CI"],
        r["Precision"] if pd.notna(r["Precision"]) else "-",
        r["Recall"] if pd.notna(r["Recall"]) else "-",
        r["FLOOR-F1"], r["Gecti_mi_Floor"],
    ])
pdf.add_table(headers, rows, col_widths)

# --- Gorsel ---
pdf.add_page()
pdf.section_title("Gorsellestirme")
img_path = os.path.join(RESULTS_DIR, "reverse_distribution_pah_cftr.png")
if os.path.exists(img_path):
    pdf.image(img_path, x=10, w=pdf.epw)

# --- Sonuc ve Karar ---
pdf.add_page()
pdf.section_title("Sonuc ve Karar")
pah_verdict = "reverse-distribution KAZANDI" if pah_delta > 0.005 else (
    "reverse-distribution NOTR/ZARARLI" if pah_delta < -0.005 else "reverse-distribution ETKISIZ (fark gurultu seviyesinde)"
)
cftr_verdict = "reverse-distribution KAZANDI" if cftr_delta > 0.005 else (
    "reverse-distribution NOTR/ZARARLI" if cftr_delta < -0.005 else "reverse-distribution ETKISIZ (fark gurultu seviyesinde)"
)
pdf.body_text(
    f"PAH: {pah_verdict}. Delta={pah_delta:+.4f}.\n"
    f"CFTR: {cftr_verdict}. Delta={cftr_delta:+.4f}. (n=21 benign nedeniyle bu deltayi tek basina "
    f"karar verici kabul etme -- CLAUDE.md'nin CFTR guven uyarisi burada da gecerli.)\n\n"
    "Literatur baglantisi: Bu deney, reports/literature_research_panel_improvements_2026-07-24.md "
    "Bolum 4, Deney #2'yi tamamlar ('Reverse-distribution'i PAH ve CFTR'ye yay'). Sonuc, yol "
    "haritasindaki bir sonraki maddeye (missing-handling panel-bazli yeniden yargilama veya TabPFN-2.5) "
    "gecis icin veri saglar."
)

report_path = os.path.join(REPORTS_DIR, "nb46_reverse_distribution_pah_cftr_report.pdf")
pdf.output(report_path)
print(f"\nPDF rapor kaydedildi: {report_path}")

In [ ]:
# Cell 11: Ozet JSON Kaydet (progress.md guncellemesi icin referans)
summary = {
    "notebook": "NB46",
    "date": datetime.now().strftime("%Y-%m-%d"),
    "hypothesis": "Reverse-distribution PAH ve CFTR'ye yayilimi",
    "pah": {
        "floor_f1": pah_floor,
        "reference_champion": {"name": "P4_COMBINED_BalBag_NB21", "boot_f1": 0.582},
        "best_reverse_dist": {
            "name": pah_best["Deney"], "boot_f1": float(pah_best["Boot-F1(prior)"]),
            "delta_vs_reference": float(pah_delta),
        },
    },
    "cftr": {
        "floor_f1": cftr_floor,
        "reference_champion": {"name": "S0c_PriorShift_NB20", "boot_f1": 0.863},
        "best_reverse_dist": {
            "name": cftr_best["Deney"], "boot_f1": float(cftr_best["Boot-F1(prior)"]),
            "delta_vs_reference": float(cftr_delta),
        },
        "caveat": "n=21 benign -- dusuk guven, CI genis",
    },
}
with open(os.path.join(RESULTS_DIR, "nb46_summary.json"), "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(json.dumps(summary, indent=2, ensure_ascii=False))